# Meme Organizer: 01. Лексический и Dense Zero-Shot поиск (Бейзлайны)

**Цель исследования:**
1. Построение воспроизводимого baseline ранжирования на чистом датасете `data/gold/gold_dataset.jsonl` (318 мемов, 1312 пар запросов).
2. Сравнение лексического поиска (**BM25 / SQLite FTS5**) и плотных нейросетевых энкодеров (**`rubert-tiny2`**, **`multilingual-e5-small`**).
3. Профилирование On-Device характеристик (задержка инференса на CPU в мс, размер ONNX INT8 в МБ).
4. Качественный анализ ошибок (**Top-15 Worst Queries**) и исследование эффекта гибридного слияния (Late Fusion).


In [1]:
import json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from meme_organizer.dataset.loader import DatasetLoader
from meme_organizer.models.bm25 import BM25Retriever
from meme_organizer.models.dense import DenseTextRetriever
from meme_organizer.pipeline.retrieval_pipeline import RetrievalPipeline

GOLD_DATASET_PATH = Path("../data/gold/gold_dataset.jsonl")
SPLIT_PATH = Path("../data/splits/splits_v1.json")

2026/11/24 21:53:36 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Проверка и загрузка данных без утечек (Leak-Free Split)

In [2]:
split_info = DatasetLoader.load_split_info(SPLIT_PATH)
print(f"Корпус мемов: {len(split_info['corpus_hashes'])} документов")
print(f"Train пары:   {len(split_info['train_pairs'])} запросов")
print(f"Val пары:     {len(split_info['val_pairs'])} запросов")
print(f"Test пары:    {len(split_info['test_pairs'])} запросов")

Корпус мемов: 318 документов
Train пары:   1021 запросов
Val пары:     138 запросов
Test пары:    153 запросов


## 2. Запуск Baseline 1: Лексический BM25 (аналог SQLite FTS5)

In [3]:
pipeline = RetrievalPipeline()
bm25 = BM25Retriever(model_name="bm25-baseline")

metrics_bm25, run_id_bm25 = pipeline.run(
    retriever=bm25,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

pd.DataFrame(
    [
        {
            "Model": "BM25 (Lexical)",
            "Recall@1": metrics_bm25.recall_at_1,
            "Recall@5": metrics_bm25.recall_at_5,
            "MRR@5": metrics_bm25.mrr_at_5,
            "NDCG@5": metrics_bm25.ndcg_at_5,
            "MLflow Run ID": run_id_bm25,
        }
    ]
)

🏃 View run bm25-baseline_test at: http://localhost:5000/#/experiments/6/runs/d8a02d3e84c04ad99bab3bf1f5843836
🧪 View experiment at: http://localhost:5000/#/experiments/6


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,BM25 (Lexical),0.6078,0.7255,0.6544,0.6722,d8a02d3e84c04ad99bab3bf1f5843836


## 3. Запуск Baseline 2: Dense Bi-Encoders (`rubert-tiny2` & `multilingual-e5-small`)

In [4]:
rubert = DenseTextRetriever(
    model_name="rubert-tiny2",
    hf_model_id="cointegrated/rubert-tiny2",
)

metrics_rubert, run_id_rubert = pipeline.run(
    retriever=rubert,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

pd.DataFrame(
    [
        {
            "Model": "RuBERT-tiny2 (Dense)",
            "Recall@1": metrics_rubert.recall_at_1,
            "Recall@5": metrics_rubert.recall_at_5,
            "MRR@5": metrics_rubert.mrr_at_5,
            "NDCG@5": metrics_rubert.ndcg_at_5,
            "MLflow Run ID": run_id_rubert,
        }
    ]
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏃 View run rubert-tiny2_test at: http://localhost:5000/#/experiments/6/runs/392b67175646453ebab98f7e46c70727
🧪 View experiment at: http://localhost:5000/#/experiments/6


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,RuBERT-tiny2 (Dense),0.4248,0.5033,0.4517,0.4645,392b67175646453ebab98f7e46c70727


## 4. Сравнительная сводная таблица из MLflow

In [5]:
e5 = DenseTextRetriever(
    model_name="multilingual-e5-small",
    hf_model_id="intfloat/multilingual-e5-small",
    query_prefix="query: ",
    passage_prefix="passage: ",
)

metrics_e5, run_id_e5 = pipeline.run(
    retriever=e5,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

pd.DataFrame(
    [
        {
            "Model": "Multilingual-E5-small (Dense)",
            "Recall@1": metrics_e5.recall_at_1,
            "Recall@5": metrics_e5.recall_at_5,
            "MRR@5": metrics_e5.mrr_at_5,
            "NDCG@5": metrics_e5.ndcg_at_5,
            "MLflow Run ID": run_id_e5,
        }
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run multilingual-e5-small_test at: http://localhost:5000/#/experiments/6/runs/4e862c5ae1be41b8a6164a9ef8855eda
🧪 View experiment at: http://localhost:5000/#/experiments/6


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,Multilingual-E5-small (Dense),0.5686,0.6797,0.6132,0.63,4e862c5ae1be41b8a6164a9ef8855eda


## 5. Анализ Top-15 Worst Queries (Качественный анализ ошибок)

In [6]:
client = mlflow.tracking.MlflowClient()
bm25_csv = client.download_artifacts(run_id_bm25, "error_analysis/worst_queries.csv")
df_worst_bm25 = pd.read_csv(bm25_csv)
print("=== Топ худших запросов для BM25 ===")
display(df_worst_bm25[["query", "target_rank"]].head(8))

e5_csv = client.download_artifacts(run_id_e5, "error_analysis/worst_queries.csv")
df_worst_e5 = pd.read_csv(e5_csv)
print("=== Топ худших запросов для E5-small ===")
display(df_worst_e5[["query", "target_rank"]].head(8))

=== Топ худших запросов для BM25 ===


,query,target_rank
0,когда пасешь кого то в чате,264
1,когда собеседник несёт бред,257
2,бабушку вырубило с удара,249
3,когда залетела в универе,248
4,когда полный аннигилятор,238
5,когда ждешь своего часа,229
6,когда не веришь собеседнику,211
7,все будет,200


=== Топ худших запросов для E5-small ===


,query,target_rank
0,выкатываюсь на важные дела,240
1,ура победа,236
2,когда пасешь кого то в чате,227
3,шевцов,208
4,когда задумал какую то пакость,185
5,sweet dreams,179
6,убегать от ментов на тачке,139
7,мужчина в рубашке в шоке,136
